In this experiment, we compare the strategy for a supervised trained energy model where, instead of predicting whether the data was transformed or not, the model predicts whether the base classifier would predict correctly. Results show that this approach improves the learned energy.

In [ ]:
import copy

import torch
import torchvision
from matplotlib import pyplot as plt



from src.utils.sampling import BatchNegativeSampler



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_emnist"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}
architecture = default_architecutre_mapping[dataset]

repeats = 5
if dataset == "tu_berlin":
    repeats = 10
if dataset == "modelnet10":
    repeats = 10
if dataset == "coil100":
    repeats = 10

In [ ]:
import seaborn as sns
plt.style.use('default')

In [ ]:
from src.utils.transforms.apply import grid_resample_border,grid_resample_reflection

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x=next(iter(test_loader_transformed))[0]

In [ ]:
batch_size = next(iter(train_loader))[0].shape[0]


In [ ]:
from src.utils.eval.vis import vis_dataset, plt_setup_latex

vis_dataset(train_loader,val_loader,test_loader_transformed)
plt.style.use('default')

In [ ]:
from model.train import train_and_get_model,train_or_load_energy_model
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "comparison_supervised_methods")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
from model.get_model import get_network, StrokeAugment

model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)

modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
#chek if data is image data
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]





from src.utils.transforms.apply import grid_resample
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)
from src.utils.replacer import replace_rotation_transforms_2vec

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:

transform_seq_non_sobol = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
).to(device)

transform_seq_sobol = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)


In [ ]:
from model.get_model import get_network_layer

layer,layer_io = get_network_layer(dataset_info, architecture, 0, num_classes=None, num_rotations=8)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence

logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)
problem_energy_logits = TransformationProblem(logit_energy, transform_seq,                                              consolidate_method="consolidate_simple")
#test ot
from search.random_search import RSLR
optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3,local_opt_kwargs={"lr":0.1})
if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60,local_runs=1,local_max_steps=0,local_opt_kwargs={"lr":0.1})


from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search, evaluate_confidence_and_search, \
    ITSWRAPPER


In [ ]:
res=load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("energy_confidence_transformed"), show_progress=True,
    repeats=repeats)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma  ,random_blur_or_sharpen,build_default_augmentations
import src.utils.augments

@torch.no_grad()
def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import build_default_augmentations, small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling_strategy
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler


energy_model2 = get_network(dataset_info,architecture, num_classes=1).to(device)



if is_image_data:
    transform_true_function = small_affine_augment_2d
    multiple_augment = src.utils.augments.build_default_augmentations()
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None
    multiple_augment = None

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=multiple_augment,
    decision_strategy=dec_strat,
)

energy_conf_default = train_or_load_energy_model(
    energy_model2, model_dir_path, f"{modelname}_energy2", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)

energy_conf_default.to(device).eval()



problem_energy_default = TransformationProblem(energy_conf_default, transform_seq,                                              consolidate_method="consolidate_simple")

res=load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_default,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_transformed"), show_progress=True,
    repeats=repeats)

#unload values
del energy_model2
del energy_conf_default
del problem_energy_default
print(res)


In [ ]:
from src.utils.augments import ComposeAugmentations, random_gaussian_noise, random_contrast, \
    random_gamma  ,random_blur_or_sharpen,build_default_augmentations
import src.utils.augments

@torch.no_grad()
def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import build_default_augmentations, small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling_strategy
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler


energy_model2 = get_network(dataset_info,architecture, num_classes=1).to(device)



if is_image_data:
    transform_true_function = small_affine_augment_2d
    multiple_augment = src.utils.augments.build_default_augmentations()
    affine_augment = ComposeAugmentations([
        lambda x: random_blur_or_sharpen(x, p=0.8, prob_blur=0.5,
                                         blur_ks_choices=(3, 5), blur_sigma_range=(0.2, 1.8),
                                         usm_ksize=5, usm_sigma_range=(0.5, 1.5),
                                         usm_amount_range=(0.5, 1.3), clamp=True),

    ])
else:
    transform_true_function = None
    affine_augment = None
    multiple_augment = None

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=multiple_augment,
    decision_strategy=dec_strat,
)

energy_conf_default = train_or_load_energy_model(
    energy_model2, model_dir_path, f"{modelname}_energy2_det", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True,deterministic_val=True)

energy_conf_default.to(device).eval()



problem_energy_default = TransformationProblem(energy_conf_default, transform_seq,                                              consolidate_method="consolidate_simple")

res=load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_default,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_transformed_det"), show_progress=True,
    repeats=repeats)

#unload values
del energy_model2
del energy_conf_default
del problem_energy_default
print(res)


In [ ]:
W =plt_setup_latex()

In [ ]:
figure_path = os.path.join(current_path,"experiment_files","export","fig","comparision_supervised",dataset,transform_name)
os.makedirs(figure_path, exist_ok=True)

In [ ]:
if is_image_data:
    x = next(iter(test_loader))[0]
    #take only 72
    x =x[:72]

    x_aug = transform_true_function(x)
    dif = (x - x_aug).abs()

    fig, axs = plt.subplots(1, 3, figsize=(W,W/2))  # 1 row, 3 columns

    # Original
    axs[0].set_title("Original")
    axs[0].imshow(torchvision.utils.make_grid(x, nrow=8).cpu().permute(1,2,0))
    axs[0].axis("off")

    # Augmented
    axs[1].set_title("Augmented")
    axs[1].imshow(torchvision.utils.make_grid(x_aug, nrow=8).cpu().permute(1,2,0))
    axs[1].axis("off")

    # Difference
    axs[2].set_title("Difference")
    axs[2].imshow(torchvision.utils.make_grid(dif, nrow=8).cpu().permute(1,2,0))
    axs[2].axis("off")


    plt.savefig(os.path.join(figure_path,"true_augmentations.png"), dpi=300,bbox_inches='tight')
    plt.tight_layout()



    plt.show()


In [ ]:
if is_image_data:
    x = next(iter(test_loader))[0]
    #take only 72
    x =x[:72]

    x_aug = affine_augment(x)
    dif = (x - x_aug).abs()

    fig, axs = plt.subplots(1, 3, figsize=(W,W/2))  # 1 row, 3 columns

    # Original
    axs[0].set_title("Original")
    axs[0].imshow(torchvision.utils.make_grid(x, nrow=8).cpu().permute(1,2,0))
    axs[0].axis("off")

    # Augmented
    axs[1].set_title("Augmented")
    axs[1].imshow(torchvision.utils.make_grid(x_aug, nrow=8).cpu().permute(1,2,0))
    axs[1].axis("off")

    # Difference
    axs[2].set_title("Difference")
    axs[2].imshow(torchvision.utils.make_grid(dif, nrow=8).cpu().permute(1,2,0))
    axs[2].axis("off")

    plt.savefig(os.path.join(figure_path,"affine_augmentations.png"), dpi=300,bbox_inches='tight')
    plt.tight_layout()
    plt.show()


In [ ]:
#if image data also test a variant without augmentations
if is_image_data:
    energy_model_no_aug = get_network(dataset_info,architecture, num_classes=1).to(device)
    negative_sampling_module_no_aug = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        =None, augment_function=None,
        decision_strategy=dec_strat,
    )

    energy_conf_no_aug = train_or_load_energy_model(
        copy.deepcopy(energy_model_no_aug), model_dir_path, f"{modelname}_energy2_no_aug", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module_no_aug, load_if_exists=True)
    energy_conf_no_aug.to(device).eval()
    problem_energy_no_aug = TransformationProblem(energy_conf_no_aug, transform_seq,                                              consolidate_method="consolidate_simple")

    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_no_aug,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_no_aug_transformed"), show_progress=True,
        repeats=repeats)

    #unload values
    del energy_model_no_aug
    del energy_conf_no_aug
    del problem_energy_no_aug
    print(res)

In [ ]:
#if image data also test a variant with only blur augmentations
if is_image_data or dataset=="tu_berlin":
    print("affine_augment")
    energy_model_no_aug = get_network(dataset_info,architecture, num_classes=1).to(device)
    negative_sampling_module_no_aug = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, mode="double_resampled"), transform_true_function
        =transform_true_function, augment_function=affine_augment,
        decision_strategy=dec_strat,
    )

    energy_conf_no_aug = train_or_load_energy_model(
        copy.deepcopy(energy_model_no_aug), model_dir_path, f"{modelname}_energy2_only_blur_aug_double_rs", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module_no_aug, load_if_exists=True)
    energy_conf_no_aug.to(device).eval()
    problem_energy_no_aug = TransformationProblem(energy_conf_no_aug, transform_seq,                                              consolidate_method="consolidate_simple")

    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_no_aug,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_only_blur_aug_transformed_double_rs"), show_progress=True,
        repeats=repeats)

    #unload values
    del energy_model_no_aug
    del energy_conf_no_aug
    del problem_energy_no_aug
    print(res)

In [ ]:
#if image data also test a variant with only blur augmentations
if is_image_data:
    energy_model_no_aug = get_network(dataset_info,architecture, num_classes=1).to(device)
    negative_sampling_module_no_aug = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        =transform_true_function, augment_function=affine_augment,
        decision_strategy=dec_strat,
    )

    energy_conf_no_aug = train_or_load_energy_model(
        copy.deepcopy(energy_model_no_aug), model_dir_path, f"{modelname}_energy2_only_blur_aug", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module_no_aug, load_if_exists=True)
    energy_conf_no_aug.to(device).eval()
    problem_energy_no_aug = TransformationProblem(energy_conf_no_aug, transform_seq,                                              consolidate_method="consolidate_simple")

    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_no_aug,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_only_blur_aug_transformed"), show_progress=True,
        repeats=repeats)

    #unload values
    del energy_model_no_aug
    del energy_conf_no_aug
    del problem_energy_no_aug
    print(res)

In [ ]:
#if image data also test a variant without augmentations
if is_image_data:
    energy_model_true_aug_only = get_network(dataset_info,architecture, num_classes=1).to(device)
    negative_sampling_module_true_aug_only = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        =transform_true_function, augment_function=None,
        decision_strategy=dec_strat,
    )
    energy_conf_true_aug_only = train_or_load_energy_model(
        copy.deepcopy(energy_model_true_aug_only), model_dir_path, f"{modelname}_energy2_true_aug_only", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module_true_aug_only, load_if_exists=True)
    energy_conf_true_aug_only.to(device).eval()
    problem_energy_true_aug_only = TransformationProblem(energy_conf_true_aug_only, transform_seq,                                              consolidate_method="consolidate_simple")
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_true_aug_only,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_true_aug_only_transformed"), show_progress=True,
        repeats=repeats)
    #unload values
    del energy_model_true_aug_only
    del energy_conf_true_aug_only
    del problem_energy_true_aug_only
    print(res)

In [ ]:
if is_image_data:
    energy_model_no_true_aug = get_network(dataset_info,architecture, num_classes=1).to(device)
    negative_sampling_module_no_true_aug = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        =None, augment_function=affine_augment,
        decision_strategy=dec_strat,
    )
    energy_conf_no_true_aug = train_or_load_energy_model(
        copy.deepcopy(energy_model_no_true_aug), model_dir_path, f"{modelname}_energy2_no_true_aug", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module_no_true_aug, load_if_exists=True)
    energy_conf_no_true_aug.to(device).eval()
    problem_energy_no_true_aug = TransformationProblem(energy_conf_no_true_aug, transform_seq,                                              consolidate_method="consolidate_simple")
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_no_true_aug,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_no_true_aug_transformed"), show_progress=True,
        repeats=repeats)
    #unload values
    del energy_model_no_true_aug
    del energy_conf_no_true_aug
    del problem_energy_no_true_aug
    print(res)

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt

import numpy as np

if is_image_data:
    res_all_augment = json.load(open(savepath("learned_energy_confidence_transformed"), "r"))
    res_no_augment = json.load(open(savepath("learned_energy_confidence_no_aug_transformed"), "r"))
    res_blur_augment = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed"), "r"))
    res_true_augment_only = json.load(open(savepath("learned_energy_confidence_true_aug_only_transformed"), "r"))
    res_blur_no_true = json.load(open(savepath("learned_energy_confidence_no_true_aug_transformed"), "r"))

    # Prepare data
    augment_types = [
        "None",
        "True",
        "Blur",
        "Both",
        "Addi-\ntional"
    ]
    means = [
        res_no_augment['accuracy_mean'],
        res_true_augment_only['accuracy_mean'],
        res_blur_no_true['accuracy_mean'],
        res_blur_augment['accuracy_mean'],
        res_all_augment['accuracy_mean']
    ]
    ses = [
        res_no_augment.get('accuracy_se', 0),
        res_true_augment_only.get('accuracy_se', 0),
        res_blur_no_true.get('accuracy_se', 0),
        res_blur_augment.get('accuracy_se', 0),
        res_all_augment.get('accuracy_se', 0)
    ]
    ses = np.array(ses)*1.96

    # Plotting
    plt.figure(figsize=(W/2,W/4))
    plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
    ax = sns.barplot(x=augment_types, y=means, palette="tab10",zorder=1,edgecolor='black',
        linewidth = 0.2,)
    plt.yticks(fontsize=8)
    plt.xticks(fontsize=8)

    # Add error bars
    ax.errorbar(x=np.arange(len(augment_types)), y=means, yerr=ses, fmt='none', c='black', capsize=3,zorder=2,        lw=0.5,
        capthick=0.5,)


    # Dynamic y-limits based on mean ﷿﷿ SE
    ymin = min([m - s for m, s in zip(means, ses)]) - 0.05
    ymax = max([m + s for m, s in zip(means, ses)]) + 0.05
    ax.set_ylim(ymin, ymax)

    ax.set_ylabel("Accuracy",fontsize=9)
    #plt.title("Comparison of Different Augmentation Strategies")
    plt.savefig(os.path.join(figure_path,"augmentation_strategies_comparison.pgf"), dpi=300,bbox_inches='tight')

    plt.savefig(os.path.join(figure_path,"augmentation_strategies_comparison.pdf"),bbox_inches='tight')
    plt.tight_layout()
    plt.show()


In [ ]:
#now test the variant that uses no dec strat
energy_model_no_dec_strat = get_network(dataset_info,architecture, num_classes=1).to(device)
negative_sampling_module_no_dec_strat = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=None,
)
energy_conf_no_dec_strat = train_or_load_energy_model(
    copy.deepcopy(energy_model_no_dec_strat), model_dir_path, f"{modelname}_energy2_no_dec_strat", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs//2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module_no_dec_strat, load_if_exists=True)
energy_conf_no_dec_strat.to(device).eval()
problem_energy_no_dec_strat = TransformationProblem(energy_conf_no_dec_strat, transform_seq,                                              consolidate_method="consolidate_simple")
res=load_or_run_evaluate_confidence_and_search(
    model, optimizer=optimizer, problem=problem_energy_no_dec_strat,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_no_dec_strat_transformed"), show_progress=True,
    repeats=repeats)
del energy_model_no_dec_strat
del energy_conf_no_dec_strat
del problem_energy_no_dec_strat
print(res)

In [ ]:
#now no dec strat no augmentations
if is_image_data:
    energy_model_no_dec_strat_no_aug = get_network(dataset_info,architecture, num_classes=1).to(device)
    negative_sampling_module_no_dec_strat_no_aug = BatchNegativeSampler(
        TransformLatentSamplingStrategy(
            transform_sequence=transform_seq, ), transform_true_function
        =None, augment_function=None,
        decision_strategy=None,
    )
    energy_conf_no_dec_strat_no_aug = train_or_load_energy_model(
        copy.deepcopy(energy_model_no_dec_strat_no_aug), model_dir_path, f"{modelname}_energy2_no_dec_strat_no_aug", train_loader,
        val_loader, trainer_kwargs={
            "accelerator": "auto",
            "max_epochs": dataset_info.epochs//2,
            "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
        }, negative_sampling_module=negative_sampling_module_no_dec_strat_no_aug, load_if_exists=True)
    energy_conf_no_dec_strat_no_aug.to(device).eval()
    problem_energy_no_dec_strat_no_aug = TransformationProblem(energy_conf_no_dec_strat_no_aug, transform_seq,                                              consolidate_method="consolidate_simple")
    res = load_or_run_evaluate_confidence_and_search(
        model, optimizer=optimizer, problem=problem_energy_no_dec_strat_no_aug,
        test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
        save_path=savepath("learned_energy_confidence_no_dec_strat_no_aug_transformed"), show_progress=True,
        repeats=repeats)
    del energy_model_no_dec_strat_no_aug
    del energy_conf_no_dec_strat_no_aug
    del problem_energy_no_dec_strat_no_aug
    print(res)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

import numpy as np

if is_image_data:
    res_blur_augment = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed"), "r"))
    res_no_dec_strat = json.load(open(savepath("learned_energy_confidence_no_dec_strat_transformed"), "r"))
    res_no_dec_strat_no_aug = json.load(open(savepath("learned_energy_confidence_no_dec_strat_no_aug_transformed"), "r"))
    res_twice_resampled = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed_double_rs"), "r"))

    loss_types = [
        "Model\nbased",
        "Model\nbased\n2x Trans.",
        "Data\nbased",
        "Data\nno-Aug"
    ]
    means = [
        res_blur_augment['accuracy_mean'],
        res_twice_resampled['accuracy_mean'],
        res_no_dec_strat['accuracy_mean'],
        res_no_dec_strat_no_aug['accuracy_mean']
    ]
    ses = [
        res_blur_augment.get('accuracy_se', 0),
        res_twice_resampled.get('accuracy_se', 0),
        res_no_dec_strat.get('accuracy_se', 0),
        res_no_dec_strat_no_aug.get('accuracy_se', 0)
    ]


elif dataset=="modelnet10":
    res_dec_strat = json.load(open(savepath("learned_energy_confidence_transformed"), "r"))
    res_no_dec_strat = json.load(open(savepath("learned_energy_confidence_no_dec_strat_transformed"), "r"))

    loss_types = [
        "Model\nbased",
        "Data\nbased"
    ]
    means = [
        res_dec_strat['accuracy_mean'],
        res_no_dec_strat['accuracy_mean']
    ]
    ses = [
        res_dec_strat.get('accuracy_se', 0),
        res_no_dec_strat.get('accuracy_se', 0)
    ]
else:
    res_dec_strat = json.load(open(savepath("learned_energy_confidence_transformed"), "r"))
    res_no_dec_strat = json.load(open(savepath("learned_energy_confidence_no_dec_strat_transformed"), "r"))
    res_twice_resampled = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed_double_rs"), "r"))

    loss_types = [
        "Model\nbased",
        "Model\nbased\n2x Trans.",
        "Data\nbased"
    ]
    means = [
        res_dec_strat['accuracy_mean'],
        res_twice_resampled['accuracy_mean'],
        res_no_dec_strat['accuracy_mean'],

    ]
    ses = [
        res_dec_strat.get('accuracy_se', 0),
        res_twice_resampled.get('accuracy_se', 0),
        res_no_dec_strat.get('accuracy_se', 0),
    ]
ses = np.array(ses)*1.96

# Plotting
plt.figure(figsize=(W/2,W/4))
plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
ax = sns.barplot(x=loss_types, y=means, palette="tab10",zorder=3,edgecolor='black',
        linewidth = 0.2,)

# Add error bars
ax.errorbar(x=np.arange(len(loss_types)), y=means, yerr=ses, fmt='none', c='black', capsize=3,zorder=4,        lw=0.5,
        capthick=0.5,)

#add more ticks
ax.yaxis.set_major_locator(plt.MaxNLocator(4))


# Dynamic y-limits based on mean ﷿﷿ SE
ymin = min([m - s for m, s in zip(means, ses)]) - 0.05
ymax = max([m + s for m, s in zip(means, ses)]) + 0.05
ax.set_ylim(ymin, ymax)

ax.set_ylabel("Accuracy",fontsize=9)
#set x tick size
ax.tick_params(axis='x', labelsize=8)


#plt.title("Comparison of Different Strategies")
plt.savefig(os.path.join(figure_path,"dec_strat_vs_no_dec_strat.pdf"), dpi=300,bbox_inches='tight')
plt.savefig(os.path.join(figure_path,"dec_strat_vs_no_dec_strat.pgf"), dpi=300,bbox_inches='tight')
plt.tight_layout()
plt.show()


In [ ]:
from src.utils.eval.vis import plt_setup_paper

W = plt_setup_paper()

In [ ]:
figure_path2 = os.path.join(current_path,"experiment_files","export","fig2","comparison_supervised",dataset,transform_name)
os.makedirs(figure_path2, exist_ok=True)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

import numpy as np

if is_image_data:
    res_blur_augment = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed"), "r"))
    res_no_dec_strat = json.load(open(savepath("learned_energy_confidence_no_dec_strat_transformed"), "r"))
    res_no_dec_strat_no_aug = json.load(open(savepath("learned_energy_confidence_no_dec_strat_no_aug_transformed"), "r"))
    res_twice_resampled = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed_double_rs"), "r"))

    loss_types = [
        "Model-based",
        "Data-based",
        "Data-based\nNo augmentations"
    ]
    means = [
        res_blur_augment['accuracy_mean'],
        res_no_dec_strat['accuracy_mean'],
        res_no_dec_strat_no_aug['accuracy_mean']
    ]
    ses = [
        res_blur_augment.get('accuracy_se', 0),
        res_no_dec_strat.get('accuracy_se', 0),
        res_no_dec_strat_no_aug.get('accuracy_se', 0)
    ]


elif dataset=="modelnet10":
    res_dec_strat = json.load(open(savepath("learned_energy_confidence_transformed"), "r"))
    res_no_dec_strat = json.load(open(savepath("learned_energy_confidence_no_dec_strat_transformed"), "r"))

    loss_types = [
        "Model\nbased",
        "Data\nbased"
    ]
    means = [
        res_dec_strat['accuracy_mean'],
        res_no_dec_strat['accuracy_mean']
    ]
    ses = [
        res_dec_strat.get('accuracy_se', 0),
        res_no_dec_strat.get('accuracy_se', 0)
    ]
else:
    res_dec_strat = json.load(open(savepath("learned_energy_confidence_transformed"), "r"))
    res_no_dec_strat = json.load(open(savepath("learned_energy_confidence_no_dec_strat_transformed"), "r"))
    res_twice_resampled = json.load(open(savepath("learned_energy_confidence_only_blur_aug_transformed_double_rs"), "r"))

    loss_types = [
        "Model\nbased",
        "Model\nbased\n2x Trans.",
        "Data\nbased"
    ]
    means = [
        res_dec_strat['accuracy_mean'],
        res_twice_resampled['accuracy_mean'],
        res_no_dec_strat['accuracy_mean'],

    ]
    ses = [
        res_dec_strat.get('accuracy_se', 0),
        res_twice_resampled.get('accuracy_se', 0),
        res_no_dec_strat.get('accuracy_se', 0),
    ]
ses = np.array(ses)*1.96

# Plotting
plt.figure(figsize=(W,W/4))
plt.grid(axis='y', linestyle=':', alpha=0.8,zorder=-1)
ax = sns.barplot(x=loss_types, y=means, palette="tab10",zorder=3,edgecolor='black',
        linewidth = 0.2,)

# Add error bars
ax.errorbar(x=np.arange(len(loss_types)), y=means, yerr=ses, fmt='none', c='black', capsize=3,zorder=4,        lw=0.5,
        capthick=0.5,)

#add more ticks
ax.yaxis.set_major_locator(plt.MaxNLocator(4))


# Dynamic y-limits based on mean ﷿﷿ SE
ymin = min([m - s for m, s in zip(means, ses)]) - 0.05
ymax = max([m + s for m, s in zip(means, ses)]) + 0.05
ax.set_ylim(ymin, ymax)

ax.set_ylabel("Accuracy",fontsize=9)
#set x tick size
ax.tick_params(axis='x', labelsize=8)


#plt.title("Comparison of Different Strategies")
plt.savefig(os.path.join(figure_path2,"dec_strat_vs_no_dec_strat.pdf"), dpi=300,bbox_inches='tight')
plt.savefig(os.path.join(figure_path2,"dec_strat_vs_no_dec_strat.pgf"), dpi=300,bbox_inches='tight')
plt.tight_layout()
plt.show()


In [ ]:
os.path.join(figure_path2,"dec_strat_vs_no_dec_strat.pgf")